## To describe the changes in Singapore's population by planning area

Region boundary is obtained from data.gov.sg

2008 and 2014 files are in SHP format, so they will be changes to gpkg for standardisation. 

`ogr2ogr -f GPKG region_boundary_2008.gpkg MasterPlan2008RegionBoundaryNoSeaSHP -nln region_boundary_2008 -nlt MULTIPOLYGON`

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import shape
from pathlib import Path
import os
from bs4 import BeautifulSoup
import re

In [2]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


In [3]:
region_boundary_2019_filepath = Path(BASE_DATASET_PATH / "onemap_data/region_boundary_2019.gpkg")
region_boundary_2019 = gpd.read_file(region_boundary_2019_filepath)

print(region_boundary_2019["Description"][1])
region_boundary_2019.columns

<center><table><tr><th colspan='2' align='center'><em>Attributes</em></th></tr><tr bgcolor="#E3E3F3"> <th>REGION_N</th> <td>NORTH REGION</td> </tr><tr bgcolor=""> <th>REGION_C</th> <td>NR</td> </tr><tr bgcolor="#E3E3F3"> <th>INC_CRC</th> <td>9642FA9C45608EBD</td> </tr><tr bgcolor=""> <th>FMEL_UPD_D</th> <td>20191223152214</td> </tr></table></center>


Index(['Name', 'Description', 'REGION_N', 'geometry'], dtype='object')

In [4]:
# extract out the region name from the Description column
def extract_region_name(description_html):
    """
    Parses the HTML-like string in the Description column to extract 
    the value associated with the 'REGION_N' key.
    
    Args:
        description_html (str): The string content from the 'Description' column.
        
    Returns:
        str: The extracted region name (e.g., 'NORTH REGION') or None if not found.
    """
    if pd.isna(description_html):
        return None
        
    try:
        # Use BeautifulSoup to parse the table structure
        soup = BeautifulSoup(description_html, 'html.parser')
        
        # Find all table rows (<tr>)
        rows = soup.find_all('tr')
        
        # Iterate through the rows to find the one containing 'REGION_N'
        for row in rows:
            # Look for the cell containing the attribute key 'REGION_N'
            # Note: The structure is <th>REGION_N</th><td>NORTH REGION</td>
            if row.find('th', string='REGION_N'):
                # The value is typically in the next cell (<td>)
                # We target the sibling <td> which holds the value
                region_name_tag = row.find('td')
                if region_name_tag:
                    return region_name_tag.text.strip()
                    
        return None # Return None if the key 'REGION_N' is not found
        
    except Exception as e:
        print(f"Error parsing description for a feature: {e}")
        return None

In [47]:
# create a new column with the region names extracted out from each row
region_boundary_2019["REGION_N"] = region_boundary_2019["Description"].apply(extract_region_name)

save_to_filepath = Path(BASE_DATASET_PATH / "onemap_data/region_boundary_2019.gpkg")

# Overwrite the layer in the GeoPackage file
# CRITICAL: mode='w' will overwrite the existing layer with the same name.
# We must ensure to use the same gpkg_file and layer name.
region_boundary_2019.to_file(
    save_to_filepath, 
    layer="region_boundary_2019", 
    driver='GPKG', 
    mode='w'  # Use 'w' (write mode) to overwrite the existing layer
)

region_boundary_2019

,Name,Description,REGION_N,geometry
0,kml_1,<center><table><tr><th colspan='2' align='cent...,WEST REGION,"MULTIPOLYGON Z (((103.74134 1.15997 0, 103.741..."
1,kml_2,<center><table><tr><th colspan='2' align='cent...,NORTH REGION,"MULTIPOLYGON Z (((103.87242 1.43995 0, 103.872..."
2,kml_3,<center><table><tr><th colspan='2' align='cent...,NORTH-EAST REGION,"MULTIPOLYGON Z (((103.94702 1.40759 0, 103.947..."
3,kml_4,<center><table><tr><th colspan='2' align='cent...,EAST REGION,"MULTIPOLYGON Z (((104.07166 1.29195 0, 104.071..."
4,kml_5,<center><table><tr><th colspan='2' align='cent...,CENTRAL REGION,"MULTIPOLYGON Z (((103.83604 1.21274 0, 103.835..."


In [90]:
population_2015_filepath = Path(BASE_DATASET_PATH / "singapore_data/data_gov/2015ResidentPopulationbyPlanningAreaSubzoneAgeGroup.csv")
population_2020_filepath = Path(BASE_DATASET_PATH / "singapore_data/singstat/2020_age_group_subzone_shorten.xlsx")

population_2015_df = pd.read_csv(population_2015_filepath)
population_2020_df = pd.read_excel(population_2020_filepath, sheet_name = "planning_area")

# make the column names all lower case
population_2015_df.columns = [x.lower() for x in population_2015_df.columns]
population_2020_df.columns = [x.lower() for x in population_2020_df.columns]

# convert all elements in the dataframe to string and lower case
population_2015_df = population_2015_df.apply(lambda x: x.astype(str).str.lower())
population_2020_df = population_2020_df.apply(lambda x: x.astype(str).str.lower())
# removing "\n"
population_2015_df['number'] = population_2015_df['number'].str.replace(r'\n', '', regex=True)
population_2020_df['planning_area'] = population_2020_df['planning_area'].str.replace(r'\n', '', regex=True)


population_2020_df.shape

(56, 21)

In [91]:
mask = population_2015_df["number"].str.contains("- total", na = False) # do not include NA
planning_area_2015 = population_2015_df[mask]
planning_area_2015.shape

(55, 58)

In [92]:
areas_2015 = planning_area_2015["number"].tolist()
areas_2015 = [s.replace('- total', '') for s in areas_2015]
areas_2015.sort()
areas_2020 = population_2020_df["planning_area"].tolist()
areas_2020 = [s.replace('\n', '') for s in areas_2020]
index_of_total = areas_2020.index("total")
del areas_2020[index_of_total]
areas_2020.sort()
print("Number of areas listed in 2015 data:", len(areas_2015))
print(areas_2015)
print("Number of areas listed in 2020 data:", len(areas_2020))
print(areas_2020)


Number of areas listed in 2015 data: 55
['ang mo kio', 'bedok', 'bishan', 'boon lay', 'bukit batok', 'bukit merah', 'bukit panjang', 'bukit timah', 'central water catchment', 'changi', 'changi bay', 'choa chu kang', 'clementi', 'downtown core', 'geylang', 'hougang', 'jurong east', 'jurong west', 'kallang', 'lim chu kang', 'mandai', 'marina east', 'marina south', 'marine parade', 'museum', 'newton', 'north-eastern islands', 'novena', 'orchard', 'outram', 'pasir ris', 'paya lebar', 'pioneer', 'punggol', 'queenstown', 'river valley', 'rochor', 'seletar', 'sembawang', 'sengkang', 'serangoon', 'simpang', 'singapore river', 'southern islands', 'straits view', 'sungei kadut', 'tampines', 'tanglin', 'tengah', 'toa payoh', 'tuas', 'western islands', 'western water catchment', 'woodlands', 'yishun']
Number of areas listed in 2020 data: 55
['ang mo kio', 'bedok', 'bishan', 'boon lay', 'bukit batok', 'bukit merah', 'bukit panjang', 'bukit timah', 'central water catchment', 'changi', 'changi bay', 

In [93]:
# if the planning areas have the same name and non of them are missing, an empty set will be returned.
missing_areas_of_areas_2015 = set(areas_2020).difference(set(areas_2015))
print(missing_areas_of_areas_2015)

missing_areas_of_areas_2020 = set(areas_2015).difference(set(areas_2020))
print(missing_areas_of_areas_2020)

{'southernislands'}
{'southern islands'}


### Southern Islands are named differently for both datasets
Checking the planning_areas obtained from onemap, Southern Islands contains spacing

Elements containing {Southern Islands, SouthernIslands} will be mapped to "Southern Islands" 

In [94]:
# The mapping dictionary
area_name_mapping = {
    'southernislands': 'southern islands',
    'Southern Islands': 'southern islands'
}

population_2020_df['planning_area'] = population_2020_df['planning_area'].map(area_name_mapping).fillna(population_2020_df['planning_area'])
# population_2020_df


#### Save the cleaned planning area data

In [ ]:
population_2015_filepath = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/2015_age_group_gender_subzone.csv")
population_2020_filepath = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/2020_age_group_subzone.xlsx")

## saving 2020 dataset to cleaned_data folder
if population_2020_filepath.exists():
    excel_mode = 'a'
    writer_kwargs = {'mode': excel_mode, 'if_sheet_exists': 'replace'} # Crucial: tells pandas to overwrite this specific sheet
    # print(f"File exists. Opening in APPEND mode ('{excel_mode}').")
else:
    excel_mode = 'w'
    writer_kwargs = {'mode': excel_mode}
    # print(f"File does not exist. Opening in WRITE mode ('{excel_mode}') to create it.")

sheet_name_to_update = "planning_area"
# select which sheet in the excel file to save to
with pd.ExcelWriter(
    population_2020_filepath, 
    engine='openpyxl',
    **writer_kwargs # Unpack the dictionary of arguments
) as writer:
    # Write the DataFrame to the specified sheet
    population_2020_df.to_excel(
        writer, 
        sheet_name=sheet_name_to_update, 
        index=False # Set to True if you want to save the DataFrame index
    )

## saving 2015 dataset to cleaned_data folder
population_2015_df.to_csv(population_2015_filepath, index = False)

### Population dataset now contains planning areas for both 2015 and 2020. 

In [103]:
# population_2015_filepath = Path(BASE_DATASET_PATH / "singapore_data/data_gov/2015ResidentPopulationbyPlanningAreaSubzoneAgeGroup.csv")
population_2020_filepath = Path(BASE_DATASET_PATH / "singapore_data/singstat/2020_age_group_subzone_shorten.xlsx")

# population_by_subzone_2015 = pd.read_csv(population_2015_filepath)
population_by_subzone_2020 = pd.read_excel(population_2020_filepath, sheet_name = "subzone")

# make the column names all lower case
# population_by_subzone_2015.columns = [x.lower() for x in population_by_subzone_2015.columns]
population_by_subzone_2020.columns = [x.lower() for x in population_by_subzone_2020.columns]

# convert all elements in the dataframe to string and lower case
# population_by_subzone_2015 = population_by_subzone_2015.apply(lambda x: x.astype(str).str.lower())
population_by_subzone_2020 = population_by_subzone_2020.apply(lambda x: x.astype(str).str.lower())

# removing "\n"
# population_by_subzone_2015['number'] = population_by_subzone_2015['number'].str.replace(r'\n', '', regex=True)
population_by_subzone_2020['subzone'] = population_by_subzone_2020['subzone'].str.replace(r'\n', '', regex=True)

# use the partially cleaned 2015 data from above
population_by_subzone_2015 = population_2015_df.copy()


In [104]:
population_by_subzone_2020.head()

,planning_area,subzone,total,0 - 4,5 - 9,10 - 14,15 - 19,20 - 24,25 - 29,30 - 34,...,45 - 49,50 - 54,55 - 59,60 - 64,65 - 69,70 - 74,75 - 79,80 - 84,85 - 89,90 & over
0,total,total,4044210,183080,198740,206390,215230,244540,287000,297800,...,311740,296070,305830,284630,229400,170010,90990,66510,36590,20880
1,ang mo kio,total,162280,5280,6100,7030,7600,8680,10320,10490,...,12410,11860,12780,12730,11960,9930,5770,4150,2280,1130
2,nan,ang mo kio town centre,4810,170,240,280,320,270,280,290,...,470,370,320,300,250,230,140,100,40,20
3,nan,cheng san,28070,1060,1040,1040,1160,1330,1710,2000,...,2200,2050,2120,2120,2170,1730,960,640,350,180
4,nan,chong boon,26500,860,840,1010,1060,1310,1610,1890,...,1820,1900,2090,2140,2100,1800,1130,780,430,200


In [105]:
def compare_subzones_rows(df_one, df_one_column: str, df_two, df_two_column: str):
    """
    To compare the rows of 2 dataframes to check for missing rows or mismatch row names

    Parameters
    ------
    - df_one: dataframe
        the first dataframe you want to compare

    - df_one_column: str
        name of the columns of the first dataframe you want to compare
    
    - df_two: dataframe
        the second dataframe you want to compare

    - df_two_column: str
        name of the columns of the second dataframe you want to compare
    """

    first_subzones = df_one[df_one_column].tolist()
    second_subzones = df_two[df_two_column].tolist()

    # removing all elements from the 2015 and 2020 list that contains the word total
    first_subzones = [item for item in first_subzones if "total" not in item]
    second_subzones = [item for item in second_subzones if "total" not in item]

    # removing \n
    cleaned_first_subzones = [s.replace('\n', '') for s in first_subzones]
    cleaned_second_subzones = [s.replace('\n', '') for s in second_subzones]

    missing_areas_of_first_subzone = set(cleaned_second_subzones).difference(set(cleaned_first_subzones))
    print(len(cleaned_first_subzones))
    # print(missing_areas_of_first_subzone)

    missing_areas_of_second_subzone = set(cleaned_first_subzones).difference(set(cleaned_second_subzones))
    print(len(cleaned_second_subzones))
    # print(missing_areas_of_second_subzone)

    return missing_areas_of_first_subzone, missing_areas_of_second_subzone


In [108]:
missing_areas_of_subzone_2015, missing_areas_of_subzone_2020 = compare_subzones_rows(population_by_subzone_2015, "number", population_by_subzone_2020, "subzone")

print(missing_areas_of_subzone_2015)
print(missing_areas_of_subzone_2020)

323
332
{'murai', 'lakeside (business)', 'park', 'nicoll', 'cleantech', 'bahar', 'tengah industrial estate', 'choa chu kangnorth', 'forest hill', 'garden', 'brickland', 'plantation', 'lakeside (leisure)'}
{'choa chu kang north', 'lakeside', 'tengah', 'western water catchment'}


### Cleaning up population dataset
Rows to remove from 2020 population dataset:
* Under tengah: brickland, forest hill, garden, park, plantation, tengah industrial estate
* Under Western Water Catchment: bahar, cleantech, murai

Changes to make to standardise the dataset:
* For Tengah in 2020 and 2015, take only the total population. 
    * **Rename Forest Hill to tengah**. Mirroring 2015's tengah subzones. The Forest Hill subzone for 2020 captures all the population in tengah.

* For Western Water Catchment in 2020 and 2015, take only the total population. 
    * **Rename Murai to western water catchment**. Mirroring 2015's western water catchment subzones. The Murai subzone for 2020 captures all the population in Western Water Catchment.

* For 2020: **change 'choa chu kangnorth' to 'choa chu kang north'**
* For 2020: combine lakeside (leisure) and lakeside (business) into lakeside, mirroring 2015's Jurong east subzones. Since lakeside (business) population is 0. 
    * Will be doing so by removing "lakeside (business)" and renameing "lakeside (leisure)" into "lakeside


In [107]:
subzones_to_remove_from_2020 = ["brickland", "garden", "park", "plantation",
                      "tengah industrial estate", "bahar", "cleantech", "lakeside (business)"]

subzone_name_mapping = {
    'choa chu kangnorth': 'choa chu kang north',
    'forest hill': 'tengah',
    'murai': 'western water catchment',
    'lakeside (leisure)': 'lakeside',
}

In [109]:
# Keep rows where Subzone names is NOT in the list of subzones to remove
filtered_subzone_2020 = population_by_subzone_2020[
    ~population_by_subzone_2020['subzone'].isin(subzones_to_remove_from_2020)
].copy()

target_column = "subzone"
# .fillna(df[target_column]) fills in NaNs (caused by unmapped values) with the original values.
filtered_subzone_2020[target_column] = filtered_subzone_2020[target_column].map(subzone_name_mapping).fillna(filtered_subzone_2020[target_column])

missing_areas_of_subzone_2015, missing_areas_of_subzone_2020 = compare_subzones_rows(population_by_subzone_2015, "number", filtered_subzone_2020, "subzone")

print("subzone name missing/mismatch from 2015 population dataset: ", missing_areas_of_subzone_2015)
print("subzone name missing/mismatch from 2020 population dataset: ", missing_areas_of_subzone_2020)

323
324
subzone name missing/mismatch from 2015 population dataset:  {'nicoll'}
subzone name missing/mismatch from 2020 population dataset:  set()


In [110]:
filtered_subzone_2020[filtered_subzone_2020["subzone"] == "nicoll"]

,planning_area,subzone,total,0 - 4,5 - 9,10 - 14,15 - 19,20 - 24,25 - 29,30 - 34,...,45 - 49,50 - 54,55 - 59,60 - 64,65 - 69,70 - 74,75 - 79,80 - 84,85 - 89,90 & over
112,nan,nicoll,290,10,10,10,10,10,30,20,...,20,30,30,30,10,20,-,-,-,-


As the nicoll subzone in downtown care is not null, it will not be removed. 
### Save the cleaned subzone data to the datasets/singapore_data/cleaned_data folder


In [111]:
population_2015_filepath = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/2015_age_group_gender_subzone.csv")
population_2020_filepath = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/2020_age_group_subzone.xlsx")

## saving 2020 dataset to cleaned_data folder
if population_2020_filepath.exists():
    excel_mode = 'a'
    writer_kwargs = {'mode': excel_mode, 'if_sheet_exists': 'replace'} # Crucial: tells pandas to overwrite this specific sheet
    # print(f"File exists. Opening in APPEND mode ('{excel_mode}').")
else:
    excel_mode = 'w'
    writer_kwargs = {'mode': excel_mode}
    # print(f"File does not exist. Opening in WRITE mode ('{excel_mode}') to create it.")

sheet_name_to_update = "subzone"
# select which sheet in the excel file to save to
with pd.ExcelWriter(
    population_2020_filepath, 
    engine='openpyxl',
    **writer_kwargs # Unpack the dictionary of arguments
) as writer:
    # Write the DataFrame to the specified sheet
    filtered_subzone_2020.to_excel(
        writer, 
        sheet_name=sheet_name_to_update, 
        index=False # Set to True if you want to save the DataFrame index
    )

## saving 2015 dataset to cleaned_data folder
population_by_subzone_2015.to_csv(population_2015_filepath, index = False)